In [2]:
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image

# ================= 配置区域 =================
# [修改] 这是一个新的数据集路径，保存校正后的张量
DATASET_ROOT = "./dataset_tri_RAD_Tensor_Corrected"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

# 真实数据路径
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "16QAM",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "64QAM",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/L20250425_103838_F6000000000_S400000_B100000_M0_IQ_ID0_nozaoshen_100ksps_QPSK.dat": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/L20250425_104306_F6000000000_S400000_B100000_M0_IQ_ID0_nojiazao_100ksps_OQPSK.dat": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

SAMPLES_PER_IMAGE = 10240 # 保持和之前一致
IMAGE_SIZE = 64
TOTAL_SIM_SAMPLES_PER_CLASS = 800
TRAIN_VAL_RATIO = 0.8
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 引入仿真函数 (保持原样) =================
# 为了脚本独立运行，这里必须包含 generate_sim_signal
# 如果你有独立的 data_generator_rad.py，可以取消下面的注释并删除这里的定义
# from data_generator_rad import generate_sim_signal

def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def apply_channel_effects(sig, snr_db):
    N = len(sig)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    phase_jitter = torch.randn(N) * 0.005 
    sig = sig * torch.exp(1j * phase_jitter)
    g_imbalance = np.random.uniform(0.98, 1.02)
    p_imbalance = np.random.uniform(-1, 1) * np.pi/180
    real = sig.real * g_imbalance
    imag = sig.imag * np.cos(p_imbalance) - sig.real * np.sin(p_imbalance)
    sig = torch.complex(real, imag)
    sig_power = torch.mean(torch.abs(sig)**2)
    real_snr = snr_db + 5 
    noise_power = sig_power / (10**(real_snr/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

def get_32qam_constellation():
    points = []
    for x in [-5, -3, -1, 1, 3, 5]:
        for y in [-5, -3, -1, 1, 3, 5]:
            if not (abs(x) == 5 and abs(y) == 5): points.append(x + 1j*y)
    return torch.tensor(points, dtype=torch.complex64)

def get_apsk_symbols(M, ratios, num_symbols):
    if M == 16:
        R1, R2 = 1.0, ratios[0]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_outer = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        constellation = torch.cat([pts_inner, pts_outer])
    elif M == 32:
        R1, R2, R3 = 1.0, ratios[0], ratios[1]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_mid = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        pts_outer = R3 * torch.exp(1j * (torch.arange(16)*2*np.pi/16))
        constellation = torch.cat([pts_inner, pts_mid, pts_outer])
    indices = torch.randint(0, M, (num_symbols,))
    return constellation[indices]

def generate_sim_signal(mod_type, N, snr_db):
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs) 
    num_symbols = int(N / sps) + 50
    syms = None
    if mod_type == "BPSK": syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK": 
        idx = torch.randint(0, 4, (num_symbols,))
        syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
    elif mod_type == "8PSK":
        idx = torch.randint(0, 8, (num_symbols,))
        syms = torch.exp(1j * idx * 2*np.pi/8)
    elif mod_type == "OQPSK":
        idx = torch.randint(0, 4, (num_symbols,))
        syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
    elif mod_type == "PI4DQPSK":
        bits = torch.randint(0, 4, (num_symbols,))
        phase_shifts = torch.tensor([np.pi/4, 3*np.pi/4, -np.pi/4, -3*np.pi/4])[bits]
        phases = torch.cumsum(phase_shifts, dim=0)
        syms = torch.exp(1j * phases)
    elif mod_type == "2ASK":
        bits = torch.randint(0, 2, (num_symbols,)).float()
        syms = bits + 0j
    elif mod_type == "16QAM":
        idx = torch.randint(0, 16, (num_symbols,))
        i = (idx // 4) * 2 - 3; q = (idx % 4) * 2 - 3; syms = i + 1j*q
    elif mod_type == "32QAM":
        constellation = get_32qam_constellation()
        idx = torch.randint(0, 32, (num_symbols,))
        syms = constellation[idx]
    elif mod_type == "64QAM":
        idx = torch.randint(0, 64, (num_symbols,))
        i = (idx // 8) * 2 - 7; q = (idx % 8) * 2 - 7; syms = i + 1j*q
    elif mod_type == "16APSK": syms = get_apsk_symbols(16, [2.7], num_symbols)
    elif mod_type == "32APSK": syms = get_apsk_symbols(32, [2.6, 4.5], num_symbols)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000) 
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        sig = apply_channel_effects(sig, snr_db)
        # FSK不进行降采样增强，或者保持一致
        return sig

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    if mod_type == "OQPSK":
        upsampled_i = torch.zeros(num_symbols * sps); upsampled_q = torch.zeros(num_symbols * sps)
        upsampled_i[::sps] = syms.real
        q_temp = torch.zeros(num_symbols * sps); q_temp[::sps] = syms.imag
        upsampled_q = torch.roll(q_temp, shifts=sps//2); upsampled_q[:sps//2] = 0
        upsampled = upsampled_i + 1j * upsampled_q
    else: upsampled[::sps] = syms

    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    
    if len(sig) > N: sig = sig[:N]
    else: sig = F.pad(sig, (0, N - len(sig)))
    
    sig = apply_channel_effects(sig, snr_db)
    # 仿真部分：加入随机降采样以模拟SPS不匹配
    step = np.random.choice([1, 2, 4])
    sig_down = sig[::step]
    if len(sig_down) < N:
        repeats = N // len(sig_down) + 1
        sig_down = sig_down.repeat(repeats)[:N] 
    else: sig_down = sig_down[:N]
    return sig_down

# ================= 2. 关键模块：强力频偏校正 =================

def correct_freq_force(iq_data):
    """
    [强力版校正]
    不设阈值，强制寻找最佳旋转角度。
    针对 QPSK (M=4) 进行优先优化。
    """
    if iq_data.dim() == 1: iq_data = iq_data.unsqueeze(0)
    
    # 使用所有数据点以获得最高频率分辨率
    n = iq_data.shape[1] 
    x = iq_data
    
    best_freq = torch.zeros(iq_data.shape[0], device=iq_data.device)
    max_power = torch.zeros(iq_data.shape[0], device=iq_data.device) - 1.0
    
    # 依次尝试 M=4 (QPSK/QAM), M=2 (BPSK), M=8 (8PSK)
    # 顺序很重要：如果 M=4 有强峰，大概率是 QPSK，应优先采纳
    for m in [4, 2, 8]:
        sig_pow = x**m
        sig_pow = sig_pow - torch.mean(sig_pow, dim=1, keepdim=True) # 去直流
        
        spectrum = torch.fft.fft(sig_pow, dim=1)
        abs_spec = torch.abs(spectrum)
        
        peak_val, peak_idx = torch.max(abs_spec, dim=1)
        
        # 简单的归一化能量作为置信度
        total_energy = torch.sum(abs_spec, dim=1) + 1e-9
        confidence = peak_val / total_energy
        
        # 如果当前 M 的置信度比之前的高，更新频偏
        # 注意：这里我们移除了 conservative threshold，只要比之前好就更新
        better_mask = confidence > max_power
        
        if better_mask.any():
            est = peak_idx.float() / float(n) / float(m)
            est = torch.where(est > 0.5/m, est - 1.0/m, est)
            
            best_freq = torch.where(better_mask, est, best_freq)
            max_power = torch.where(better_mask, confidence, max_power)

    # 生成校正信号
    t = torch.arange(n, device=iq_data.device)
    correction = torch.exp(-1j * 2 * np.pi * best_freq.unsqueeze(1) * t)
    
    return (iq_data * correction).squeeze()

# ================= 3. RAD生成逻辑 =================

def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]; y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max); y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def generate_tensor_data(iq_complex_data, img_size=64):
    if iq_complex_data.dim() == 1:
        iq_complex_data = iq_complex_data.unsqueeze(0)
    iq_data = iq_complex_data.squeeze()
    
    if iq_data.numel() < 100: 
        return torch.zeros(3, img_size, img_size)

    # 归一化
    scale = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    norm_iq = iq_data / (3.0 * scale)
    
    # 提取特征
    real = norm_iq.real
    imag = norm_iq.imag
    phase = torch.angle(norm_iq) / np.pi 
    mag = torch.abs(norm_iq)
    
    diff_signal = norm_iq[1:] * torch.conj(norm_iq[:-1])
    d_scale = torch.sqrt(torch.mean(diff_signal.real**2 + diff_signal.imag**2)) + 1e-9
    diff_signal = diff_signal / (3.0 * d_scale)
    d_real = diff_signal.real
    d_imag = diff_signal.imag
    
    # 生成直方图
    bins = img_size
    hist_r = fast_hist2d(real, imag, bins=bins, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    hist_a = fast_hist2d(phase, mag, bins=bins, range_bounds=((-1.0, 1.0), (0.0, 1.5)))
    hist_d = fast_hist2d(d_real, d_imag, bins=bins, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    def process(h):
        h = torch.log1p(h)
        mx = h.max()
        if mx > 1e-6: h = h / mx
        return h

    tensor = torch.stack([process(hist_r), process(hist_a), process(hist_d)], dim=0)
    return torch.flip(tensor, [1])

# ================= 执行生成 =================
if __name__ == "__main__":

    if os.path.exists(DATASET_ROOT):
        shutil.rmtree(DATASET_ROOT)
    
    print("===== Generating Frequency Corrected Tensor Data (.pt) =====")
    print("Logic: Simulate/Read -> FORCE Correct Freq -> Generate RAD -> Save")
    
    # 1. 仿真数据生成
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        os.makedirs(os.path.join(SIM_VAL_DIR, cls_name), exist_ok=True)
        
        print(f"Generating Sim {cls_name}...")
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            target_dir = SIM_TRAIN_DIR if np.random.rand() < TRAIN_VAL_RATIO else SIM_VAL_DIR
            target_dir = os.path.join(target_dir, cls_name)
            
            snr = np.random.uniform(-5, 25)
            # 1. 生成带频偏的仿真信号
            sig = generate_sim_signal(cls_name, SAMPLES_PER_IMAGE, snr)
            if isinstance(sig, np.ndarray): sig = torch.from_numpy(sig)
            sig = sig.to(DEVICE)

            # 2. [关键] 强制频偏校正 (模拟实测处理)
            # 我们希望训练集包含 "校正后依然有残留误差" 的数据
            sig_corrected = correct_freq_force(sig)

            # 3. 生成张量
            tensor_data = generate_tensor_data(sig_corrected, IMAGE_SIZE)
            
            save_path = os.path.join(target_dir, f"{cls_name}_{i:04d}_{snr:.1f}.pt")
            torch.save(tensor_data.cpu(), save_path)
            
            if i == 0:
                from PIL import Image
                arr = (tensor_data.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
                Image.fromarray(arr).save(os.path.join(target_dir, "viz_sample.png"))

    # 2. 实测数据处理
    print("\nProcessing Real Data...")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): 
            print(f"Skip {label}: File not found")
            continue
            
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        try:
            # 兼容读取
            if file_path.endswith('.dat'):
                raw = np.fromfile(file_path, dtype='>i2')
                if len(raw) > 1024: raw = raw[1024:]
            else:
                raw = np.fromfile(file_path, dtype=np.int16)
            
            if len(raw) % 2 != 0: raw = raw[:-1]
            
            # 分离实虚部 (防止 ComplexWarning)
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            
            # 组合成 PyTorch 复数
            iq_complex = torch.complex(torch.from_numpy(I), torch.from_numpy(Q))
            iq_complex = iq_complex / 32767.0
            iq_complex = iq_complex.to(DEVICE)
            
            num_slices = len(iq_complex) // SAMPLES_PER_IMAGE
            saved_count = 0 
            
            for i in range(num_slices):
                if saved_count >= 500: break
                
                chunk = iq_complex[i*SAMPLES_PER_IMAGE : (i+1)*SAMPLES_PER_IMAGE]
                
                # 1. [关键] 对实测切片进行强制频偏校正
                chunk_corrected = correct_freq_force(chunk)
                
                # 2. 生成 RAD 张量
                tensor_data = generate_tensor_data(chunk_corrected, IMAGE_SIZE)
                
                # 保存
                torch.save(tensor_data.cpu(), os.path.join(save_dir, f"real_{i:04d}.pt"))
                saved_count += 1
                
        except Exception as e:
            print(f"Error {label}: {e}")
            import traceback
            traceback.print_exc()
            
    print(f"\nDone! Corrected Dataset saved at {DATASET_ROOT}")

===== Generating Frequency Corrected Tensor Data (.pt) =====
Logic: Simulate/Read -> FORCE Correct Freq -> Generate RAD -> Save
Generating Sim BPSK...


100%|██████████| 800/800 [00:03<00:00, 215.00it/s]


Generating Sim QPSK...


100%|██████████| 800/800 [00:03<00:00, 206.31it/s]


Generating Sim 8PSK...


100%|██████████| 800/800 [00:04<00:00, 191.58it/s]


... (15 outputs left out)

100%|██████████| 800/800 [00:03<00:00, 208.74it/s]


Generating Sim FSK2...


100%|██████████| 800/800 [00:02<00:00, 281.43it/s]



Processing Real Data...

Done! Corrected Dataset saved at ./dataset_tri_RAD_Tensor_Corrected
